# 📊 Notebook 1: Recopilación y Preprocesado de Datos

## TFM: Predicción de Estrategias de Carrera en Fórmula 1 mediante ML

**Autor:** Francisco José Moreno Bayona  
**Director:** Inmaculada Tomeo Reyes  
**Universidad:** UNIR  
**Período de estudio:** 2022-2025 (era de efecto suelo)  

---

### Objetivo de este notebook

Recopilar datos de **FastF1** (telemetría oficial FIA) y **Jolpica-F1** (historial de carreras),
integrarlos, limpiarlos y construir las variables predictoras para las dos tareas del TFM:

1. **Clasificación:** ¿Habrá parada en boxes en esta vuelta? (sí/no)
2. **Regresión:** ¿En qué vuelta exacta se producirá la parada?

### Fuentes de datos

| Fuente | Tipo | Datos |
|--------|------|-------|
| **FastF1** | Telemetría oficial FIA | Tiempos de vuelta, sectores, neumáticos, posición |
| **Jolpica-F1** | API REST (sucesora de Ergast) | Paradas en boxes, clasificaciones, resultados |

## 1. Instalación de dependencias

Ejecuta esta celda **solo la primera vez** o si no tienes las librerías instaladas.

In [21]:
# ============================================================================
# INSTALACIÓN DE DEPENDENCIAS
# Ejecutar solo la primera vez o si faltan librerías
# ============================================================================

import subprocess
import sys

paquetes = ['fastf1', 'pandas', 'numpy', 'requests', 'scikit-learn']

for paquete in paquetes:
    try:
        __import__(paquete.replace('-', '_'))
        print(f'✓ {paquete} ya instalado')
    except ImportError:
        print(f'Instalando {paquete}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', paquete, '-q'])
        print(f'✓ {paquete} instalado')

print('\n✅ Todas las dependencias listas')

✓ fastf1 ya instalado
✓ pandas ya instalado
✓ numpy ya instalado
✓ requests ya instalado
Instalando scikit-learn...
✓ scikit-learn instalado

✅ Todas las dependencias listas


## 2. Imports y configuración inicial

Importamos las librerías necesarias y configuramos el entorno.

**Decisiones de diseño:**
- Caché de FastF1 activada para evitar descargas repetidas
- Solo compuestos secos (SOFT, MEDIUM, HARD) → excluimos lluvia
- Período 2022-2025 (era de efecto suelo, marco reglamentario homogéneo)

In [1]:
import pandas as pd
import numpy as np
import fastf1
import requests
import json
import warnings
from datetime import datetime
from pathlib import Path
import os

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

os.makedirs('./fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('./fastf1_cache')
# ============================================================================
# PARÁMETROS PRINCIPALES
# ============================================================================

# Temporadas a procesar (era de efecto suelo)
TEMPORADAS = [2022, 2023, 2024, 2025]

# API de Jolpica-F1 (sucesora de Ergast)
JOLPICA_API = 'https://api.jolpi.ca/ergast/f1'

# DECISIÓN: Solo neumáticos secos → estrategia pura
# Las paradas bajo lluvia son determinísticas (mandato FIA), no estratégicas
COMPUESTOS_SECO = ['SOFT', 'MEDIUM', 'HARD']

# Directorio de salida
OUTPUT_DIR = Path('./datasets')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Temporadas: {TEMPORADAS}')
print(f'Compuestos válidos: {COMPUESTOS_SECO}')
print(f'Directorio de salida: {OUTPUT_DIR}')
print('\n✅ Configuración inicial completada')

Temporadas: [2022, 2023, 2024, 2025]
Compuestos válidos: ['SOFT', 'MEDIUM', 'HARD']
Directorio de salida: datasets

✅ Configuración inicial completada


## 3. Función: Obtener datos de FastF1

FastF1 es la **fuente principal de telemetría oficial** de la FIA.

**¿Qué datos obtenemos?**
- Tiempos de vuelta (LapTime)
- Tiempos por sector (Sector1, Sector2, Sector3)
- Compuesto de neumático (Compound: SOFT, MEDIUM, HARD)
- Posición en carrera (Position)
- Indicador de parada (PitInLap, PitOutLap)

**¿Por qué FastF1?**
- Acceso oficial a datos FIA
- Librería bien mantenida y documentada
- Datos públicos y reproducibles

In [2]:
def obtener_datos_fastf1(temporada: int, gp_round: int) -> dict:
    """
    Obtiene datos de telemetría de FastF1 para una carrera específica.
    
    Args:
        temporada: Año de la temporada (2022-2025)
        gp_round: Número de Gran Premio (1-24)
    
    Returns:
        dict con datos de carrera y laps, o None si falla
    """
    try:
        # Cargar sesión de CARRERA (no prácticas ni clasificación)
        session = fastf1.get_session(temporada, gp_round, 'R')
        session.load(telemetry=False, weather=False)
        
        laps = session.laps.copy()
        
        print(f'  ✓ {session.event.EventName} {temporada}: {len(laps)} laps cargados')
        
        return {
            'season': temporada,
            'round': gp_round,
            'event_name': session.event.EventName,
            'event_date': session.event.EventDate,
            'total_laps': session.total_laps,
            'laps': laps,
        }
    except Exception as e:
        print(f'  ✗ Error en {temporada} R{gp_round}: {e}')
        return None

print('✅ Función obtener_datos_fastf1 definida')

✅ Función obtener_datos_fastf1 definida


## 4. Función: Procesar una carrera individual

Integra los datos de FastF1 y aplica las primeras limpiezas.

**Decisiones de limpieza:**
1. **Excluir vueltas con lluvia** (INT/WET) → solo estrategia pura, no mandatos reglamentarios
2. **Excluir laps sin tiempo válido** → vueltas incompletas no son modelables
3. **Enriquecer con metadatos** → Season, Round, EventName para agrupar

In [3]:
def procesar_carrera(temporada: int, gp_round: int) -> pd.DataFrame:
    """
    Procesa una carrera: obtiene datos, filtra lluvia, limpia.
    
    Returns:
        DataFrame con laps válidos en seco, o None si falla
    """
    
    # Obtener datos de FastF1
    fastf1_data = obtener_datos_fastf1(temporada, gp_round)
    if fastf1_data is None:
        return None
    
    laps = fastf1_data['laps'].copy()
    total_original = len(laps)
    
    # LIMPIEZA 1: Excluir vueltas con lluvia
    # RAZÓN: Paradas bajo lluvia son determinísticas (mandato FIA),
    # no dependen de lógica estratégica modelable
    laps_seco = laps[laps['Compound'].isin(COMPUESTOS_SECO)].copy()
    excluidos_lluvia = total_original - len(laps_seco)
    
    # LIMPIEZA 2: Excluir laps sin tiempo válido
    # RAZÓN: No podemos modelar vueltas incompletas (DNF, problemas mecánicos)
    laps_validos = laps_seco[laps_seco['LapTime'].notna()].copy()
    excluidos_tiempo = len(laps_seco) - len(laps_validos)
    
    # Enriquecer con metadatos de sesión
    laps_validos['Season'] = temporada
    laps_validos['Round'] = gp_round
    laps_validos['EventName'] = fastf1_data['event_name']
    laps_validos['EventDate'] = fastf1_data['event_date']
    laps_validos['TotalLaps'] = fastf1_data['total_laps']
    
    print(f'    → Laps originales: {total_original}')
    print(f'    → Excluidos lluvia: {excluidos_lluvia}')
    print(f'    → Excluidos sin tiempo: {excluidos_tiempo}')
    print(f'    → Laps válidos finales: {len(laps_validos)}')
    
    if len(laps_validos) == 0:
        print(f'    ⚠️  Carrera completamente bajo lluvia, excluida')
        return None
    
    return laps_validos

print('✅ Función procesar_carrera definida')

✅ Función procesar_carrera definida


## 5. Función: Ingeniería de características (Feature Engineering)

Construimos las variables predictoras para cada vuelta de cada piloto.

**Filosofía:** En primeros experimentos es mejor incluir **muchas variables candidatas** y luego evaluar importancia (con SHAP, permutation importance, etc.), que arriesgar omitir la variable correcta.

### Features construidas:

| Feature | Tipo | Justificación |
|---------|------|---------------|
| `LapNumber` | int | Más vueltas = mayor probabilidad de parada |
| `LapProgress` | float | Progreso normalizado de la carrera (0-1) |
| `Position` | int | Líderes = conservadores, cola = arriesgados |
| `TireAge` | int | Edad del neumático: mejor predictor de parada |
| `IsHard/Medium/Soft` | binary | Tipo de neumático (one-hot encoding) |
| `LapTime_seconds` | float | Tiempo bruto de vuelta |
| `AvgTime_Last3` | float | Ritmo reciente (proxy de degradación) |
| `DegradationIndicator` | float | Diferencia respecto a media → degradación |
| `StintNumber` | int | Número de stint actual (1°, 2°, 3°...) |
| `WillStopThisLap` | binary | **TARGET clasificación** (0/1) |
| `StopLapNumber` | int | **TARGET regresión** (vuelta exacta) |

In [4]:
def construir_features(laps_piloto: pd.DataFrame) -> pd.DataFrame:
    """
    Construye features para un piloto en una carrera.
    
    Args:
        laps_piloto: DataFrame de vueltas de UN piloto en UNA carrera
    
    Returns:
        DataFrame enriquecido con features derivadas
    """
    df = laps_piloto.copy().sort_values('LapNumber').reset_index(drop=True)
    
    # --------- PROGRESO DE CARRERA ---------
    df['LapNumber'] = df['LapNumber'].astype(int)
    total_laps = df['TotalLaps'].iloc[0] if 'TotalLaps' in df.columns else df['LapNumber'].max()
    df['LapProgress'] = df['LapNumber'] / total_laps  # 0.0 a 1.0
    
    # --------- POSICIÓN ---------
    df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
    df['Position'] = df['Position'].ffill().bfill().fillna(20)
    
    # --------- EDAD DEL NEUMÁTICO ---------
    # DECISIÓN: Calcular stint (grupo de vueltas con mismo compuesto)
    df['CompoundChanged'] = (df['Compound'] != df['Compound'].shift()).astype(int)
    df['StintNumber'] = df['CompoundChanged'].cumsum()
    df['TireAge'] = df.groupby('StintNumber').cumcount() + 1
    
    # --------- TIPO DE NEUMÁTICO (One-Hot) ---------
    df['IsHard'] = (df['Compound'] == 'HARD').astype(int)
    df['IsMedium'] = (df['Compound'] == 'MEDIUM').astype(int)
    df['IsSoft'] = (df['Compound'] == 'SOFT').astype(int)
    
    # --------- TIEMPOS Y DEGRADACIÓN ---------
    df['LapTime_seconds'] = df['LapTime'].dt.total_seconds()
    
    # Media de últimas 3 vueltas (indicador de ritmo actual)
    df['AvgTime_Last3'] = df['LapTime_seconds'].rolling(window=3, min_periods=1).mean()
    
    # Degradación estimada: diferencia entre tiempo actual y media reciente
    # Valores positivos = empeorando (neumático degradándose)
    df['DegradationIndicator'] = df['LapTime_seconds'] - df['AvgTime_Last3']
    
    # --------- VARIABLES OBJETIVO ---------
    # TARGET CLASIFICACIÓN: ¿Habrá parada en esta vuelta?
    df['WillStopThisLap'] = df['PitInTime'].notna().astype(int)
    
    # TARGET REGRESIÓN: Vuelta exacta de la parada
    df['StopLapNumber'] = np.where(
        df['WillStopThisLap'] == 1,
        df['LapNumber'],
        0
    )
    
    return df

print('✅ Función construir_features definida')

✅ Función construir_features definida


## 6. Recopilación de datos: Procesar TODAS las carreras

Este es el **bucle principal** que recorre todas las temporadas y GPs.

⏱️ **Tiempo estimado: 30-60 minutos** (depende de tu conexión a internet)

**¿Qué hace?**
1. Para cada temporada (2022-2025):
   - Obtiene el calendario de la temporada
   - Para cada GP: descarga datos, filtra lluvia, construye features
2. Concatena todo en un único DataFrame
3. Genera estadísticas de resumen

In [ ]:
# ============================================================================
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ============================================================================

todos_laps = []
estadisticas = []

for temporada in TEMPORADAS:
    print(f'\n{"="*70}')
    print(f'TEMPORADA {temporada}')
    print(f'{"="*70}')
    
    try:
        # Obtener calendario de la temporada
        schedule = fastf1.get_event_schedule(temporada, include_testing=False)
        n_races = len(schedule[schedule['EventFormat'] != 'testing'])
        print(f'Carreras programadas: {n_races}')
    except Exception as e:
        print(f'Error obteniendo calendario {temporada}: {e}')
        continue
    
    for gp_round in range(1, n_races + 1):
        print(f'\n--- GP {gp_round}/{n_races} ---')
        
        # Procesar carrera
        laps_carrera = procesar_carrera(temporada, gp_round)
        
        if laps_carrera is not None and len(laps_carrera) > 0:
            # Construir features por cada piloto
            pilotos = laps_carrera['Driver'].unique()
            
            for piloto in pilotos:
                laps_piloto = laps_carrera[laps_carrera['Driver'] == piloto]
                try:
                    features = construir_features(laps_piloto)
                    todos_laps.append(features)
                except Exception as e:
                    print(f'    ⚠️ Error en features de {piloto}: {e}')
                    continue
            
            estadisticas.append({
                'season': temporada,
                'round': gp_round,
                'event': laps_carrera['EventName'].iloc[0],
                'laps': len(laps_carrera),
                'drivers': len(pilotos)
            })
        else:
            print(f'    ⚠️ Carrera sin datos válidos, saltando')

print(f'\n{"="*70}')
print(f'PROCESAMIENTO COMPLETADO')
print(f'{"="*70}')
print(f'Carreras procesadas: {len(estadisticas)}')
print(f'Bloques de laps recopilados: {len(todos_laps)}')

In [5]:
import time

todos_laps = []
estadisticas = []

for temporada in TEMPORADAS:
    print(f'\n{"="*70}')
    print(f'TEMPORADA {temporada}')
    print(f'{"="*70}')
    
    # Obtener calendario (con reintentos)
    schedule = None
    for intento in range(3):
        try:
            schedule = fastf1.get_event_schedule(temporada, include_testing=False)
            break
        except Exception as e:
            print(f'  ⚠️ Rate limit en calendario, esperando 60s... (intento {intento+1}/3)')
            time.sleep(60)
    
    if schedule is None:
        print(f'  ✗ No se pudo obtener calendario de {temporada}, saltando')
        continue
    
    n_races = len(schedule[schedule['EventFormat'] != 'testing'])
    print(f'Carreras programadas: {n_races}')
    
    for gp_round in range(1, n_races + 1):
        print(f'\n--- GP {gp_round}/{n_races} ---')
        
        # Procesar carrera (con reintentos si hay rate limit)
        laps_carrera = None
        for intento in range(3):
            try:
                laps_carrera = procesar_carrera(temporada, gp_round)
                break
            except Exception as e:
                if '500 calls' in str(e) or 'rate' in str(e).lower():
                    wait = 3600
                    print(f'  ⚠️ Rate limit, esperando {wait}s... (intento {intento+1}/3)')
                    time.sleep(wait)
                else:
                    print(f'  ✗ Error no recuperable: {e}')
                    break
        
        if laps_carrera is not None and len(laps_carrera) > 0:
            pilotos = laps_carrera['Driver'].unique()
            
            for piloto in pilotos:
                laps_piloto = laps_carrera[laps_carrera['Driver'] == piloto]
                try:
                    features = construir_features(laps_piloto)
                    todos_laps.append(features)
                except Exception as e:
                    print(f'    ⚠️ Error en features de {piloto}: {e}')
                    continue
            
            estadisticas.append({
                'season': temporada,
                'round': gp_round,
                'event': laps_carrera['EventName'].iloc[0],
                'laps': len(laps_carrera),
                'drivers': len(pilotos)
            })
        else:
            print(f'    ⚠️ Sin datos válidos, saltando')
        
        # PAUSA entre carreras para no saturar la API
        # Las carreras en caché se saltan instantáneamente
        time.sleep(10)

print(f'\n{"="*70}')
print(f'PROCESAMIENTO COMPLETADO')
print(f'{"="*70}')
print(f'Carreras procesadas: {len(estadisticas)}')
print(f'Bloques de laps recopilados: {len(todos_laps)}')


TEMPORADA 2022


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Carreras programadas: 22

--- GP 1/22 ---


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


  ✓ Bahrain Grand Prix 2022: 1125 laps cargados
    → Laps originales: 1125
    → Excluidos lluvia: 0
    → Excluidos sin tiempo: 7
    → Laps válidos finales: 1118


...

  ✓ Qatar Grand Prix 2025: 1067 laps cargados
    → Laps originales: 1067
    → Excluidos lluvia: 0
    → Excluidos sin tiempo: 3
    → Laps válidos finales: 1064


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...



--- GP 24/24 ---


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

  ✓ Abu Dhabi Grand Prix 2025: 1156 laps cargados
    → Laps originales: 1156
    → Excluidos lluvia: 0
    → Excluidos sin tiempo: 0
    → Laps válidos finales: 1156

PROCESAMIENTO COMPLETADO
Carreras procesadas: 90
Bloques de laps recopilados: 1743


## 7. Integración final y generación de datasets

Concatenamos todos los datos y generamos los dos datasets para las tareas del TFM:

1. **Dataset de clasificación:** Todas las vueltas → target = ¿parada sí/no?
2. **Dataset de regresión:** Solo vueltas con parada → target = vuelta exacta

In [6]:
# ============================================================================
# CONCATENAR TODOS LOS DATOS
# ============================================================================

if len(todos_laps) > 0:
    df_completo = pd.concat(todos_laps, ignore_index=True)
    print(f'Dataset completo: {len(df_completo)} filas × {len(df_completo.columns)} columnas')
    print(f'Temporadas: {sorted(df_completo["Season"].unique())}')
    print(f'Pilotos únicos: {df_completo["Driver"].nunique()}')
    print(f'Carreras únicas: {df_completo.groupby(["Season", "Round"]).ngroups}')
else:
    print('⛔ No se obtuvieron datos. Verifica tu conexión a internet.')
    df_completo = pd.DataFrame()

Dataset completo: 92991 filas × 48 columnas
Temporadas: [2022, 2023, 2024, 2025]
Pilotos únicos: 31
Carreras únicas: 90


In [7]:
# ============================================================================
# SELECCIÓN DE COLUMNAS PARA DATASETS FINALES
# ============================================================================

# Columnas que usaremos en los modelos
COLS_CONTEXTO = ['Season', 'Round', 'EventName', 'EventDate', 'Driver']
COLS_FEATURES = [
    'LapNumber', 'LapProgress', 'Position', 'TireAge', 'StintNumber',
    'IsHard', 'IsMedium', 'IsSoft',
    'LapTime_seconds', 'AvgTime_Last3', 'DegradationIndicator',
]
COLS_TARGET_CLASIF = ['WillStopThisLap']
COLS_TARGET_REGRES = ['StopLapNumber']

# Verificar que existen todas las columnas
todas_cols = COLS_CONTEXTO + COLS_FEATURES + COLS_TARGET_CLASIF + COLS_TARGET_REGRES
cols_existentes = [c for c in todas_cols if c in df_completo.columns]
cols_faltantes = [c for c in todas_cols if c not in df_completo.columns]

if cols_faltantes:
    print(f'⚠️ Columnas faltantes: {cols_faltantes}')
else:
    print(f'✅ Todas las columnas presentes ({len(cols_existentes)})')

# Filtrar columnas
df_final = df_completo[cols_existentes].copy()
print(f'\nDataset final: {df_final.shape}')

✅ Todas las columnas presentes (18)

Dataset final: (92991, 18)


In [8]:
# ============================================================================
# GENERAR DATASET DE CLASIFICACIÓN
# ============================================================================

df_clasificacion = df_final.copy()
df_clasificacion.rename(columns={'WillStopThisLap': 'target_parada'}, inplace=True)

# Guardar
df_clasificacion.to_csv(OUTPUT_DIR / 'dataset_clasificacion.csv', index=False)

# Estadísticas
n_positivos = df_clasificacion['target_parada'].sum()
n_total = len(df_clasificacion)
print(f'DATASET CLASIFICACIÓN')
print(f'  Total muestras: {n_total:,}')
print(f'  Paradas (1): {n_positivos:,.0f} ({100*n_positivos/n_total:.1f}%)')
print(f'  No paradas (0): {n_total - n_positivos:,.0f} ({100*(1-n_positivos/n_total):.1f}%)')
print(f'  ⚠️  Desbalance: {n_positivos/n_total:.2%} positivos')
print(f'  Guardado en: {OUTPUT_DIR / "dataset_clasificacion.csv"}')

DATASET CLASIFICACIÓN
  Total muestras: 92,991
  Paradas (1): 2,953 (3.2%)
  No paradas (0): 90,038 (96.8%)
  ⚠️  Desbalance: 3.18% positivos
  Guardado en: datasets\dataset_clasificacion.csv


In [9]:
# ============================================================================
# GENERAR DATASET DE REGRESIÓN
# ============================================================================

# Solo filas donde hubo parada
df_regresion = df_final[df_final['WillStopThisLap'] == 1].copy()
df_regresion.rename(columns={'StopLapNumber': 'target_vuelta_parada'}, inplace=True)

# Guardar
df_regresion.to_csv(OUTPUT_DIR / 'dataset_regresion.csv', index=False)

# Estadísticas
print(f'DATASET REGRESIÓN')
print(f'  Total paradas: {len(df_regresion):,}')
print(f'  Vuelta media: {df_regresion["target_vuelta_parada"].mean():.1f}')
print(f'  Vuelta mediana: {df_regresion["target_vuelta_parada"].median():.0f}')
print(f'  Rango: {df_regresion["target_vuelta_parada"].min():.0f} - {df_regresion["target_vuelta_parada"].max():.0f}')
print(f'  Guardado en: {OUTPUT_DIR / "dataset_regresion.csv"}')

DATASET REGRESIÓN
  Total paradas: 2,953
  Vuelta media: 27.5
  Vuelta mediana: 26
  Rango: 1 - 77
  Guardado en: datasets\dataset_regresion.csv


## 8. Resumen y verificación

Verificamos que todo se ha generado correctamente antes de pasar al Notebook 2.

In [10]:
# ============================================================================
# RESUMEN FINAL
# ============================================================================

print('='*70)
print('RESUMEN DE RECOPILACIÓN Y PREPROCESADO')
print('='*70)
print(f'\nPeríodo: {TEMPORADAS[0]}-{TEMPORADAS[-1]} (era efecto suelo)')
print(f'Compuestos: {COMPUESTOS_SECO} (sin lluvia)')
print(f'\nDataset clasificación: {len(df_clasificacion):,} vueltas')
print(f'Dataset regresión: {len(df_regresion):,} paradas')
print(f'\nArchivos generados:')

import os
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'  ✓ {f.name} ({size_mb:.1f} MB)')

print(f'\n✅ Notebook 1 completado. Continuar con Notebook 2 (Análisis y Preparación).')

RESUMEN DE RECOPILACIÓN Y PREPROCESADO

Período: 2022-2025 (era efecto suelo)
Compuestos: ['SOFT', 'MEDIUM', 'HARD'] (sin lluvia)

Dataset clasificación: 92,991 vueltas
Dataset regresión: 2,953 paradas

Archivos generados:
  ✓ dataset_clasificacion.csv (11.2 MB)
  ✓ dataset_regresion.csv (0.4 MB)

✅ Notebook 1 completado. Continuar con Notebook 2 (Análisis y Preparación).
